# Cleaning raw (scrapped) data from "TMDB"
`https://www.kaggle.com/datasets/kakarlaramcharan/tmdb-data-0920`

In [1]:
!pip install gradio -q
!pip install faiss-cpu --no-cache -q
!pip install transformers -q
!pip install datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.9/322.9 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 129.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 271.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 16.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not

In [2]:
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from google.colab import files
import gradio as gr
import pandas as pd
import numpy as np
import faiss
import ast

In [4]:
from huggingface_hub import notebook_login
notebook_login()

In [3]:
# Load raw data
df_raw = pd.read_csv("movie_data_tmbd.csv", sep="|", engine='python', on_bad_lines='skip', quoting=1)

# Remove some columns
df_dropcols = df_raw.drop(columns=[
    "adult", "backdrop_path", "belongs_to_collection", "budget", "homepage", "imdb_id", "runtime", "id", "popularity", "poster_path", "production_companies", "production_countries", "release_date", "revenue", "status", "tagline", "video", "vote_average", "vote_count", "cast", "directors"
    ], axis=1)

# Select only the english ones
df_lang = df_dropcols[df_dropcols["original_language"] == "en"]
df_clean = df_lang.copy()

# Create a function to extract the "genres" and "spoken_lenguages"
def extract_names(value):
    try:
        items = ast.literal_eval(value)
        return [item['name'] for item in items if 'name' in item]
    except (ValueError, SyntaxError):
        return []

# Extract names
df_clean["genres"] = df_clean["genres"].apply(extract_names)
df_clean["spoken_languages"] = df_clean["spoken_languages"].apply(extract_names)

# Delete "[]" and spaces
df_clean["genres"] = df_clean["genres"].astype(str).str.strip("[]").str.replace("'", "").str.replace(", ", ",")
df_clean["spoken_languages"] = df_clean["spoken_languages"].astype(str).str.strip("[]").str.replace("'", "").str.replace(", ", ",")

# Separate columns
df_clean[['genre_1', 'genre_2']] = df_clean['genres'].str.split(',', n=1,expand=True)
df_clean[['language_1', 'language_2']] = df_clean['spoken_languages'].str.split(',', n=1, expand=True)

# Fill with "None"
df_clean[['genre_1', 'genre_2']] = df_clean[['genre_1', 'genre_2']].fillna("None")
df_clean[['language_1', 'language_2']] = df_clean[['language_1', 'language_2']].fillna("None")

# Delete the originals
df_clean = df_clean.drop(columns=["genres", "spoken_languages"])
df_clean = df_clean.dropna()

# Delete the 2nd (to much null values)
df_clean = df_clean.drop(columns=["genre_2", "language_2"])

# Fill the null
df_clean["language_1"] = df_clean["language_1"].replace("", "English")
df_clean["genre_1"] = df_clean["genre_1"].replace("", "None")

# Top 5 languages
df_final = df_clean[df_clean["language_1"].isin(["English", "Español", "Deutsch", "Français", "Italiano"])]

# Export
df_final.to_csv("movie_data_final.csv", index=False)

## Creating a Dataset from the clean data.

In [4]:
raw_dataset = load_dataset("csv", data_files=r"/content/movie_data_final.csv")

split_dataset = raw_dataset.map(
    lambda x: {"comment_length": len(x["overview"].split())}
)

filter_dataset = split_dataset.filter(lambda x: x["comment_length"] > 15)

def concatenate_text(examples):
    for key in ["title", "genre_1", "language_1", "overview"]:
        if examples.get(key) is None:
            examples[key] = ""

    return {
        "text": examples["title"]
        + " \n "
        + examples["genre_1"]
        + " \n "
        + examples["language_1"]
        + " \n "
        + examples["overview"]
    }


concat_data = filter_dataset.map(concatenate_text)

final_dataset = concat_data.remove_columns(["original_title", "original_language", "comment_length"])
final_dataset = final_dataset.rename_columns({
    "genre_1" : "genre",
    "language_1": "language"
})

final_dataset.save_to_disk("final_dataset")
final_dataset

In [5]:
final_dataset.push_to_hub("clean-embedding-movies", commit_message="Second commit")

## Creating a movie recommendation system with Embeddings >>>>  NLP + text similarity

In [3]:
dataset = load_dataset("asfilcnx3/clean-embedding-movies")
dataset

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/420 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/28.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/56946 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['overview', 'title', 'genre', 'language', 'text'],
        num_rows: 56946
    })
})

In [4]:
# Load model to semantic search with FAISS
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
texts = dataset["train"]["text"]
embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)

dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1780 [00:00<?, ?it/s]

In [7]:
## Interactive Demo on GRADIO
# Function to found the index from the title
def get_index_from_title(title_query, titles_list):
    try:
        return titles_list.index(title_query)
    except ValueError:
        return None

# Recomend by title
def recommend_by_title(title_query, titles_list, faiss_index, embeddings, top_k=5):
    title_query = title_query.lower().strip()
    idx = get_index_from_title(title_query, titles_list)
    if idx is None:
        return "Movie not found. Please check the title and try again."

    # recomendation with FAISS
    _, indices = faiss_index.search(embeddings[idx:idx+1], top_k + 1)
    similar_titles = [titles_list[i] for i in indices[0] if i != idx]

    return "\n".join(similar_titles[:top_k])

# Title list
titles_list = [title.lower() for title in dataset["train"]["title"]]

# Gradio interface
demo = gr.Interface(
    fn=lambda title: recommend_by_title(title, titles_list, faiss_index, embeddings),
    inputs=gr.Textbox(label="Enter a movie title"),
    outputs=gr.Textbox(label="Recommended Movies"),
    title="Movie Recommender",
    description="Type the title of a movie and get 5 similar recommendations based on text embeddings.",
    examples= [["Interstellar"], ["The Dark Knight"], ["Alien"]]
)

demo.launch()

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://34839676d5e2c29eeb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
